# Smoke test: `model.py`

In [1]:
import sys
import numpy as np
import torch

# Ensure the `snn_research` package is importable regardless of nbconvert CWD.
sys.path.insert(0, r"C:\Users\sriha\Downloads\SNN")

from snn_research.data import state_to_spikes
from snn_research.model import SimpleANN, SimpleSNN, QuinticPlanner, SNNController

# SimpleANN forward (expects MNIST-like input)
ann = SimpleANN()
x = torch.zeros((2, 1, 28, 28), dtype=torch.float32)
logits, spike_count = ann(x)
print("SimpleANN logits shape:", tuple(logits.shape), "spikes:", spike_count)

# SimpleSNN forward
snn = SimpleSNN(num_steps=5)
logits_snn, spike_count_snn = snn(x)
print("SimpleSNN logits shape:", tuple(logits_snn.shape), "total_spikes:", spike_count_snn)

# QuinticPlanner callable
planner = QuinticPlanner(T=1.0, dt=0.1)
action0, flops0 = planner(np.array([1.0, 0.0], dtype=np.float32))
action1, flops1 = planner(np.array([0.5, -0.1], dtype=np.float32))
print("QuinticPlanner actions:", action0, action1, "flops:", flops0, flops1)

# SNNController forward + STDP smoke update
ctrl = SNNController(num_steps=3)
spike_vec = state_to_spikes(np.array([0.2, 0.0, -0.1, 0.05], dtype=np.float32))
spike_t = torch.tensor([spike_vec], dtype=torch.float32)
logits_c, sc = ctrl.forward(spike_t)
print("SNNController logits shape:", tuple(logits_c.shape), "spike_count:", sc)

# Quick STDP update smoke test with all-ones spike tensors
# Use asymmetric pre/post timing so LTP != LTD cancel out.
pre = [torch.ones_like(ctrl.stdp_weights)] * 1
post = [torch.ones_like(ctrl.stdp_weights)] * 2
w_before = ctrl.stdp_weights.detach().clone()
ctrl.apply_stdp(pre, post)
w_after = ctrl.stdp_weights.detach().clone()
diff_max = float((w_before - w_after).abs().max().item())
print("SNNController STDP max |Δw|:", diff_max)
print("SNNController STDP updated weights:", diff_max > 1e-12)

print("MODEL SMOKE TEST PASSED")


SimpleANN logits shape: (2, 10) spikes: 0
SimpleSNN logits shape: (2, 10) total_spikes: tensor(0., grad_fn=<AddBackward0>)
QuinticPlanner actions: 0 1 flops: 20 20
SNNController logits shape: (1, 2) spike_count: 8
SNNController STDP max |Δw|: 0.00951230525970459
SNNController STDP updated weights: True
MODEL SMOKE TEST PASSED
